In [1]:
import sys; 
sys.path.append("..")
import pandas as pd, numpy as np
from src.data.utils import load_config, PROJECT_ROOT

cfg = load_config()
df  = pd.read_csv(PROJECT_ROOT / cfg["data"]["raw_path"])
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Fraud rate: {df.FraudFound_P.mean():.2%}")


Dataset loaded: 15420 rows, 33 columns
Fraud rate: 5.99%


In [2]:
fraud    = df[df.FraudFound_P == 1]
legit    = df[df.FraudFound_P == 0]
print("--- Days_Policy_Accident (fraud vs legit) ---")
print(pd.crosstab(df.Days_Policy_Accident, df.FraudFound_P, normalize="columns").round(3))
print("\n--- PastNumberOfClaims (fraud vs legit) ---")
print(pd.crosstab(df.PastNumberOfClaims, df.FraudFound_P, normalize="columns").round(3))


--- Days_Policy_Accident (fraud vs legit) ---
FraudFound_P              0      1
Days_Policy_Accident              
1 to 7                0.001  0.001
15 to 30              0.003  0.003
8 to 15               0.003  0.005
more than 30          0.989  0.980
none                  0.003  0.010

--- PastNumberOfClaims (fraud vs legit) ---
FraudFound_P            0      1
PastNumberOfClaims              
1                   0.231  0.241
2 to 4              0.358  0.319
more than 4         0.134  0.074
none                0.277  0.367


In [3]:
print(df.FraudFound_P.value_counts())
print(f"Fraud ratio: {df.FraudFound_P.mean():.4f}")
print(f"This means contamination should be set to approximately: {df.FraudFound_P.mean():.2f}")


FraudFound_P
0    14497
1      923
Name: count, dtype: int64
Fraud ratio: 0.0599
This means contamination should be set to approximately: 0.06


In [ ]:
import sys; 
sys.path.append("..")
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from src.data.utils import load_config, PROJECT_ROOT
import pandas as pd


cfg = load_config()
features = pd.read_csv(PROJECT_ROOT / cfg["data"]["features_path"])


y = features.pop("FraudFound_P").values
X = StandardScaler().fit_transform(features.values)

results = []
for c in [0.01, 0.03, 0.06, 0.10, 0.15, 0.20]:
    iso = IsolationForest(contamination=c, n_estimators=100, random_state=42)
    iso.fit(X)
    pred = (iso.predict(X) == -1).astype(int)
    results.append({
        "contamination": c,
        "flagged": pred.sum(),
        "precision": precision_score(y, pred),
        "recall":    recall_score(y, pred),
        "f1":        f1_score(y, pred)
    })
print(pd.DataFrame(results).round(3).to_string(index=False))


 contamination  flagged  precision  recall    f1
          0.01      155      0.071   0.012 0.020
          0.03      463      0.065   0.033 0.043
          0.06      926      0.063   0.063 0.063
          0.10     1542      0.064   0.106 0.080
          0.15     2313      0.064   0.159 0.091
          0.20     3084      0.061   0.205 0.094


In [9]:
features_leaky = pd.read_csv(PROJECT_ROOT / cfg["data"]["features_path"])   # FraudFound_P still in features for leaky model
X_leaky = StandardScaler().fit_transform(features_leaky.values)
iso_leaky = IsolationForest(contamination=0.06, n_estimators=100, random_state=42)
iso_leaky.fit(X_leaky)
pred_leaky = (iso_leaky.predict(X_leaky) == -1).astype(int)
print("Leaky F1:", round(f1_score(features_leaky["FraudFound_P"], pred_leaky), 3))

Leaky F1: 0.131
